# Medical QA — RAG Pipeline (V1–V4) — Colab Runner

**Course:** NLP Applications in Research and Industry, Uni Mainz SoSe 2026
**Author:** Peter Altmayer

---

## What this notebook does
Runs the full V1–V4 extractive QA comparison across three datasets (BioASQ, PubMedQA, SQuAD)
with BM25 and dense retrieval, then prints a summary table + failure analysis.

| Phase | What happens | Estimated time (T4 GPU) |
|---|---|---|
| 0 — Colab setup | Mount Drive, install packages, clone repo | ~5 min |
| 1 — Verify data | Check BioASQ JSON is on Drive | < 1 min |
| 2 — Build indexes | BM25 + FAISS for all 3 datasets | ~5 min |
| 3a — Train V1 | 2-layer BERT from scratch on SQuAD-10k | ~20 min |
| 3b — Train V3 | `bert-base-uncased` FT on BioASQ+PubMedQA | ~30 min |
| 4 — Evaluate | All 4 models × 3 datasets × 2 retrievers | ~60 min |
| 5 — Results | Summary table + failure analysis | < 1 min |

**Total first run: ~2 hours.**  
Indexes and checkpoints are stored on Google Drive and survive session resets — subsequent runs skip completed phases automatically.

---
### Before you start
1. **Enable GPU**: Runtime → Change runtime type → Hardware accelerator → **T4 GPU** → Save
2. **Upload BioASQ**: see Phase 1 below

---
## Phase 0 — Colab Setup
Run cells 0-A through 0-D once per session (they are fast and idempotent).

In [ ]:
# 0-A  Mount Drive + verify GPU
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import torch, subprocess
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if DEVICE == 'cuda':
    gpu = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        capture_output=True, text=True
    ).stdout.strip()
    print(f'GPU  : {gpu}')
else:
    print('NO GPU detected.')
    print('Go to: Runtime > Change runtime type > T4 GPU, then re-run.')

print(f'PyTorch : {torch.__version__}')
print(f'Device  : {DEVICE}')

In [ ]:
# 0-B  Install packages
import sys

!{sys.executable} -m pip install -q \
    "transformers>=4.40" \
    "datasets>=2.18" \
    "sentence-transformers>=2.7" \
    "rank-bm25>=0.2.2" \
    "rouge-score>=0.1.2" \
    "accelerate>=0.27" \
    faiss-cpu tqdm pandas numpy scikit-learn

print('Packages ready.')

In [ ]:
# 0-C  Clone repo + set working directory
import os, sys

GITHUB_REPO = 'https://github.com/peter-altmayer/NLP_Applications_in_Research_and_Industry_SoSe_26.git'
REPO_ROOT   = '/content/NLP_Applications_in_Research_and_Industry_SoSe_26'
PROJECT_DIR = f'{REPO_ROOT}/nlp_medical_qa'

if not os.path.exists(REPO_ROOT):
    print('Cloning repo...')
    os.system(f'git clone --depth 1 {GITHUB_REPO} {REPO_ROOT}')
else:
    print('Repo already present — pulling latest...')
    # Drive symlinks at data/raw, data/processed, results conflict with .gitkeep files
    # in the repo. Remove them before pulling; cell 0-D recreates them.
    for _rel in ['data/raw', 'data/processed', 'results']:
        _lp = f'{PROJECT_DIR}/{_rel}'
        if os.path.islink(_lp):
            os.unlink(_lp)
    os.system(f'git -C {REPO_ROOT} pull --quiet')

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(f'Working dir : {os.getcwd()}')

In [ ]:
# 0-D  Link Google Drive storage into the project
#      Indexes, checkpoints and results are saved here and persist across sessions.
import shutil

DRIVE_BASE = '/content/drive/MyDrive/nlp_medical_qa_data'

for sub in ['data/raw', 'data/processed', 'results']:
    os.makedirs(f'{DRIVE_BASE}/{sub}', exist_ok=True)

def _link(local_rel, drive_sub):
    local  = f'{PROJECT_DIR}/{local_rel}'
    target = f'{DRIVE_BASE}/{drive_sub}'
    os.makedirs(target, exist_ok=True)
    if os.path.islink(local):
        print(f'  (already linked) {local_rel}/ -> Drive')
        return
    if os.path.exists(local):
        shutil.rmtree(local)
    # Ensure parent directory exists (may be absent if .gitkeep wasn't cloned)
    os.makedirs(os.path.dirname(local), exist_ok=True)
    os.symlink(target, local)
    print(f'  Linked: {local_rel}/ -> {target}')

_link('data/raw',       'data/raw')
_link('data/processed', 'data/processed')
_link('results',        'results')

print(f'\nDrive base: {DRIVE_BASE}')

---
## Phase 1 — BioASQ Data

You need `training13b.json` on Google Drive. **Do this once — it persists.**

### Option A — Upload via Drive web UI (recommended)
1. Open [drive.google.com](https://drive.google.com)
2. Navigate to `nlp_medical_qa_data/data/raw/`
3. Create folder `BioASQ-training13b`
4. Upload `training13b.json` into that folder
5. Run the verify cell below

### Option B — Upload directly in this notebook
Run cell 1-A below, which opens a file picker.

### Option C — Direct download URL
If BioASQ gave you a download link:
```python
!wget -q -O '/content/drive/MyDrive/nlp_medical_qa_data/data/raw/BioASQ-training13b/training13b.json' 'YOUR_URL'
```

In [ ]:
# 1-A  OPTIONAL: upload BioASQ JSON via file picker
#      Skip this cell if you already put the file on Drive via the web UI.
BIOASQ_TARGET = f'{DRIVE_BASE}/data/raw/BioASQ-training13b/training13b.json'

if os.path.exists(BIOASQ_TARGET):
    print(f'BioASQ already on Drive: {BIOASQ_TARGET}')
    print('Skip this cell.')
else:
    from google.colab import files as colab_files
    print('Select training13b.json in the file picker...')
    uploaded = colab_files.upload()
    for fname, data in uploaded.items():
        dest_dir = f'{DRIVE_BASE}/data/raw/BioASQ-training13b'
        os.makedirs(dest_dir, exist_ok=True)
        dest = f'{dest_dir}/training13b.json'
        with open(dest, 'wb') as fh:
            fh.write(data)
        print(f'Saved -> {dest} ({len(data)//1024//1024} MB)')

In [ ]:
# 1-B  Verify BioASQ + show stats
import json

BIOASQ_JSON = f'{DRIVE_BASE}/data/raw/BioASQ-training13b/training13b.json'

if not os.path.exists(BIOASQ_JSON):
    print(f'BioASQ JSON not found at:\n  {BIOASQ_JSON}')
    print('Run cell 1-A or upload via Drive web UI.')
elif os.path.getsize(BIOASQ_JSON) == 0:
    print(f'ERROR: BioASQ JSON exists but is EMPTY (0 bytes):\n  {BIOASQ_JSON}')
    print('The upload likely failed or was interrupted.')
    print('Please re-upload training13b.json via cell 1-A or the Drive web UI.')
else:
    with open(BIOASQ_JSON, encoding='utf-8') as f:
        _d = json.load(f)
    _qs = _d['questions']
    _by_type = {}
    for q in _qs:
        _by_type[q['type']] = _by_type.get(q['type'], 0) + 1

    print(f'BioASQ 13b: {len(_qs)} questions total')
    for t, n in sorted(_by_type.items()):
        used = 'USED' if t in ('factoid', 'yesno') else 'skipped'
        print(f'  {t:10s} {n:4d}  [{used}]')
    print(f'\nUsable for eval  (factoid+yesno): {_by_type.get("factoid",0)+_by_type.get("yesno",0)}')
    print(f'Usable for V3 FT (yesno only)   : {_by_type.get("yesno",0)}')
    print('\nBioASQ data OK.')

---
## Phase 2 — Build Retrieval Indexes
Builds BM25 and FAISS indexes for BioASQ, PubMedQA, and SQuAD.  
**Runs once; output saved to Drive. Skipped automatically on subsequent runs.**

In [ ]:
# 2  Build indexes
# IMPORTANT: if you pulled new code (corpus now includes test split + ideal_answers),
# delete the old stale indexes first — run cell X-3 below, then re-run this cell.
PROCESSED = f'{DRIVE_BASE}/data/processed'

missing_ds  = set()
missing_ret = set()
for ds in ['bioasq', 'pubmedqa', 'squad']:
    for ret, chk in [('bm25', f'{PROCESSED}/bm25_{ds}.pkl'),
                     ('dense', f'{PROCESSED}/dense_{ds}.faiss')]:
        if not os.path.exists(chk):
            missing_ds.add(ds)
            missing_ret.add(ret)

if not missing_ds:
    print('All indexes already on Drive.')
    print('If results look wrong, delete stale indexes with cell X-3 and re-run.')
else:
    print(f'Building indexes for datasets : {sorted(missing_ds)}')
    print(f'Retrieval types               : {sorted(missing_ret)}')
    print('Corpus = train + test splits + ideal_answers (~10 min)...')
    ret = os.system(
        f'python experiments/build_index.py '
        f'--datasets {" ".join(sorted(missing_ds))} '
        f'--retrieval {" ".join(sorted(missing_ret))}'
    )
    if ret == 0:
        print('Indexes saved to Drive.')
    else:
        print('Index build failed — check output above.')

---
## Phase 3 — Train V1 and V3

V2 and V4 are downloaded directly from HuggingFace and need no training.  
V1 and V3 must be trained first; checkpoints are saved to Drive.

| Model | Training data | Time |
|---|---|---|
| **V1** (from scratch) | SQuAD train, 10 000 examples, 3 epochs | ~20 min |
| **V3** (fine-tune BERT) | BioASQ yesno + PubMedQA, ~2 000 examples, 4 epochs | ~30 min |

Each cell is **safe to re-run** — if a checkpoint already exists on Drive it will be skipped.

In [ ]:
# 3-A  Train V1 (2-layer BERT from scratch)
V1_CKPT = f'{DRIVE_BASE}/data/processed/checkpoints/v1'

if os.path.exists(f'{V1_CKPT}/config.json'):
    print(f'V1 checkpoint already on Drive: {V1_CKPT}')
    print('Skipping training.')
else:
    print('Training V1 from scratch (~20 min)...\n')
    ret = os.system('python experiments/run_experiment.py --model v1 --fine-tune')
    if ret == 0:
        print('\nV1 checkpoint saved to Drive.')
    else:
        print('\nV1 training failed — check output above.')

In [ ]:
# 3-B  Fine-tune V3 (bert-base-uncased on biomedical QA data)
V3_CKPT = f'{DRIVE_BASE}/data/processed/checkpoints/v3'

if os.path.exists(f'{V3_CKPT}/config.json'):
    print(f'V3 checkpoint already on Drive: {V3_CKPT}')
    print('Skipping training.')
else:
    print('Fine-tuning V3 on BioASQ yesno + PubMedQA (~30 min)...\n')
    ret = os.system('python experiments/run_experiment.py --model v3 --fine-tune')
    if ret == 0:
        print('\nV3 checkpoint saved to Drive.')
    else:
        print('\nV3 training failed — check output above.')

---
## Phase 4 — Evaluate All Models

Each combination `model × dataset × retrieval` is evaluated on 200 samples and saves a CSV to Drive.  
Already-completed runs are skipped.

**Options:**
- **Cell 4-A** — Run a single combination (good for testing)
- **Cell 4-B** — Run the full matrix: 4 models × 3 datasets × 2 retrievers = 24 runs (~60 min)

Add `--skip-faithfulness` to skip the NLI faithfulness step and cut ~40% of runtime.

In [ ]:
# 4-A  Run a single experiment (edit as needed)
# Good for a quick sanity check before the full run.
import subprocess

MODEL     = 'v2'       # v1 | v2 | v3 | v4
DATASET   = 'bioasq'  # bioasq | pubmedqa | squad
RETRIEVAL = 'dense'   # dense | bm25
K         = 5
N         = 1          # samples (use 1-20 for quick test, 200 for real run)

OUT_CSV = f'{DRIVE_BASE}/results/{MODEL}_{DATASET}_{RETRIEVAL}_k{K}.csv'
if os.path.exists(OUT_CSV):
    print(f'Result already exists: {OUT_CSV}')
    print('Delete it from Drive to re-run, or change MODEL/DATASET/RETRIEVAL above.')
else:
    cmd = [
        sys.executable, 'experiments/run_experiment.py',
        '--model', MODEL, '--dataset', DATASET,
        '--retrieval', RETRIEVAL, '--k', str(K), '--n', str(N),
        '--skip-faithfulness',
    ]
    print(f'Running: {" ".join(cmd)}\n')
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        print(f'Failed with exit code {result.returncode}')

In [ ]:
# 4-B  Full evaluation matrix  (4 models x 3 datasets x 2 retrievers)
# Already-completed runs are skipped; V1/V3 runs are skipped if no checkpoint exists.
import subprocess

MODELS     = ['v1', 'v2', 'v3', 'v4']
DATASETS   = ['bioasq', 'pubmedqa', 'squad']
RETRIEVALS = ['dense', 'bm25']
K = 5
N = 200          # reduce to 50 for a faster full run
SKIP_FAITH = False  # False = run faithfulness scoring | True = skip it (~40% faster)

results_dir = f'{DRIVE_BASE}/results'
ckpt_dir    = f'{DRIVE_BASE}/data/processed/checkpoints'
faith_flag  = ['--skip-faithfulness'] if SKIP_FAITH else []

# Check which trainable models have checkpoints
trained = {
    'v1': os.path.exists(f'{ckpt_dir}/v1/config.json'),
    'v3': os.path.exists(f'{ckpt_dir}/v3/config.json'),
}
if not trained['v1']:
    print('WARNING: V1 checkpoint missing — run cell 3-A first to train V1.')
if not trained['v3']:
    print('WARNING: V3 checkpoint missing — run cell 3-B first to train V3.')

total = len(MODELS) * len(DATASETS) * len(RETRIEVALS)
done  = 0

for model in MODELS:
    for dataset in DATASETS:
        for retrieval in RETRIEVALS:
            done += 1
            label = f'[{done:2d}/{total}] {model} | {dataset} | {retrieval}'

            # Skip trainable models without checkpoints
            if model in ('v1', 'v3') and not trained[model]:
                print(f'{label}  SKIP (no checkpoint — run cell 3-{"A" if model=="v1" else "B"} first)')
                continue

            out_csv = f'{results_dir}/{model}_{dataset}_{retrieval}_k{K}.csv'
            if os.path.exists(out_csv):
                print(f'{label}  SKIP (already done)')
                continue

            print(f'{label}  running...')
            cmd = [
                sys.executable, 'experiments/run_experiment.py',
                '--model', model, '--dataset', dataset,
                '--retrieval', retrieval, '--k', str(K), '--n', str(N),
            ] + faith_flag
            result = subprocess.run(cmd, capture_output=True, text=True)
            if result.stdout:
                print(result.stdout)
            if result.returncode != 0:
                print(f'  ERROR (exit {result.returncode}):')
                if result.stderr:
                    for line in result.stderr.strip().split('\n')[-30:]:
                        print(f'    {line}')
                print(f'  WARNING: run failed, continuing...')

print('\nAll evaluation runs complete.')

In [ ]:
# Re-run V4 only — V1/V2/V3 CSVs already saved, will be skipped
import subprocess, sys, os

DRIVE_BASE = '/content/drive/MyDrive/nlp_medical_qa_data'
PROJECT_DIR = '/content/NLP_Applications_in_Research_and_Industry_SoSe_26/nlp_medical_qa'
K, N = 5, 200

for dataset in ['bioasq', 'pubmedqa', 'squad']:
    for retrieval in ['dense', 'bm25']:
        out_csv = f'{DRIVE_BASE}/results/v4_{dataset}_{retrieval}_k{K}.csv'
        if os.path.exists(out_csv):
            print(f'SKIP v4 | {dataset} | {retrieval} (already done)')
            continue
        print(f'Running v4 | {dataset} | {retrieval} ...')
        cmd = [sys.executable, 'experiments/run_experiment.py',
                '--model', 'v4', '--dataset', dataset,
                '--retrieval', retrieval, '--k', str(K), '--n', str(N)]
        result = subprocess.run(cmd, capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print('ERROR:', result.stderr[-500:])

---
## Phase 5 — Results
Loads all result CSVs from Drive, prints the summary table, and shows failure examples.

In [ ]:
# 5-A  Summary metrics table (pandas display)
import pandas as pd
from pathlib import Path
import glob

results_dir = f'{DRIVE_BASE}/results'
csv_files   = [f for f in glob.glob(f'{results_dir}/*.csv') if 'summary' not in f]

if not csv_files:
    print('No result CSVs found. Run Phase 4 first.')
else:
    METRIC_COLS = ['em', 'f1', 'rouge_l', 'bertscore', 'faithfulness',
                   'precision_at_k', 'recall_at_k']

    dfs = []
    for p in csv_files:
        df = pd.read_csv(p)
        dfs.append(df)
    all_data = pd.concat(dfs, ignore_index=True)

    for col in METRIC_COLS:
        all_data[col] = pd.to_numeric(all_data[col], errors='coerce')

    summary = (
        all_data
        .groupby(['model', 'dataset', 'retrieval'])[METRIC_COLS]
        .mean()
        .round(3)
        .reset_index()
    )

    # Order models V1 -> V4
    model_order = {'v1': 0, 'v2': 1, 'v3': 2, 'v4': 3}
    summary['_order'] = summary['model'].map(model_order)
    summary = summary.sort_values(['dataset', 'retrieval', '_order']).drop(columns='_order')

    pd.set_option('display.max_columns', 20)
    pd.set_option('display.width', 120)
    pd.set_option('display.float_format', '{:.3f}'.format)

    for dataset in summary['dataset'].unique():
        print(f'\n=== Dataset: {dataset.upper()} ===')
        display(summary[summary['dataset'] == dataset].reset_index(drop=True))

    # Save summary CSV to Drive
    out = f'{results_dir}/summary.csv'
    summary.to_csv(out, index=False)
    print(f'\nSummary saved -> {out}')

In [ ]:
# 5-B  Failure analysis — worst predictions per model x dataset
TOP_N = 3  # worst examples to show per group

if not csv_files:
    print('No result CSVs found. Run Phase 4 first.')
else:
    print(f'=== FAILURE ANALYSIS — {TOP_N} worst samples per model x dataset ===')

    for (model, dataset), group in all_data.groupby(['model', 'dataset']):
        worst = group.dropna(subset=['f1']).nsmallest(TOP_N, 'f1')
        if worst.empty:
            continue
        print(f'\n--- {model.upper()} / {dataset} ---')
        for _, row in worst.iterrows():
            q   = str(row.get('question', ''))[:90]
            gld = str(row.get('gold_answer', ''))[:60]
            prd = str(row.get('predicted_answer', ''))[:60]
            f1  = row.get('f1', float('nan'))
            em  = row.get('em', float('nan'))
            fa  = row.get('faithfulness', None)
            fa_str = f'{fa:.3f}' if fa is not None and fa == fa else '—'
            print(f'  Q    : {q}')
            print(f'  Gold : {gld}')
            print(f'  Pred : {prd}')
            print(f'  F1={f1:.3f}  EM={em:.0f}  Faithfulness={fa_str}')
            print()

In [ ]:
# 5-C  Bar chart: F1 by model across datasets
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

if not csv_files:
    print('No data yet.')
else:
    # Use dense retrieval for the plot
    plot_data = summary[summary['retrieval'] == 'dense'].copy()
    if plot_data.empty:
        plot_data = summary.copy()

    datasets  = sorted(plot_data['dataset'].unique())
    models    = ['v1', 'v2', 'v3', 'v4']
    models    = [m for m in models if m in plot_data['model'].unique()]
    x         = range(len(datasets))
    width     = 0.18
    colors    = ['#e15759', '#4e79a7', '#f28e2b', '#59a14f']

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for ax, metric, title in [
        (axes[0], 'f1',        'Token F1'),
        (axes[1], 'bertscore', 'BERTScore'),
    ]:
        for i, (model, color) in enumerate(zip(models, colors)):
            vals = [
                plot_data.loc[
                    (plot_data['model'] == model) & (plot_data['dataset'] == ds), metric
                ].values[0] if len(plot_data.loc[
                    (plot_data['model'] == model) & (plot_data['dataset'] == ds)
                ]) > 0 else 0
                for ds in datasets
            ]
            pos = [xi + (i - len(models)/2 + 0.5) * width for xi in x]
            ax.bar(pos, vals, width, label=model.upper(), color=color, alpha=0.85)

        ax.set_xticks(list(x))
        ax.set_xticklabels([d.upper() for d in datasets])
        ax.set_ylabel(title)
        ax.set_title(f'{title} by Model (dense retrieval, k=5)')
        ax.legend()
        ax.set_ylim(0, 1)
        ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plot_path = f'{DRIVE_BASE}/results/v1_v4_comparison.png'
    plt.savefig(plot_path, bbox_inches='tight')
    plt.show()
    print(f'Plot saved -> {plot_path}')

In [ ]:
# 5-D  Retrieval comparison: BM25 vs Dense (for best model V4)
if not csv_files:
    print('No data yet.')
else:
    ret_cmp = summary[summary['model'] == 'v4'][['dataset', 'retrieval', 'precision_at_k', 'recall_at_k', 'f1']]
    if ret_cmp.empty:
        ret_cmp = summary[['model', 'dataset', 'retrieval', 'precision_at_k', 'recall_at_k', 'f1']]

    print('=== Retrieval ablation (BM25 vs Dense) ===')
    display(ret_cmp.sort_values(['dataset', 'retrieval']).reset_index(drop=True))

---
## Extras — Utilities
Useful one-off cells.

In [ ]:
# X-1  Interactive: ask a question and see what V4 + dense retrieval answers
import sys
sys.path.insert(0, PROJECT_DIR)

from src.model import load_model
from src.retriever import DenseRetriever
from pathlib import Path

# Load V4 model and BioASQ dense index
_model     = load_model('v4', device=DEVICE)
_retriever = DenseRetriever.load(Path(PROJECT_DIR) / 'data' / 'processed' / 'dense_bioasq')

QUESTION = 'Is the protein Papilin secreted?'  # <-- change this

retrieved = _retriever.retrieve(QUESTION, k=5)
context   = ' '.join(retrieved)
answer    = _model.predict(QUESTION, context)

print(f'Question : {QUESTION}')
print(f'Answer   : {answer}')
print(f'\nTop-1 source passage:')
print(f'  {retrieved[0][:300]}')

In [ ]:
# X-2  Show Drive storage usage
import os

def _dir_size_mb(path):
    total = 0
    for dirpath, _, fnames in os.walk(path):
        for f in fnames:
            fp = os.path.join(dirpath, f)
            try:
                total += os.path.getsize(fp)
            except OSError:
                pass
    return total / 1024 / 1024

for sub in ['data/raw', 'data/processed', 'results']:
    p = f'{DRIVE_BASE}/{sub}'
    if os.path.exists(p):
        print(f'{sub:20s} {_dir_size_mb(p):7.1f} MB')

print(f'\nDrive base: {DRIVE_BASE}')

In [ ]:
# X-3  Purge stale indexes, checkpoints and results (run before re-building after a code update)
# Deletes BM25/FAISS indexes, V1/V3 checkpoints, and all result CSVs from Drive.
# Run Phase 2, Phase 3, and Phase 4 again afterwards.
import glob, shutil

confirm = input("Type YES to delete all indexes, checkpoints and result CSVs: ")
if confirm.strip().upper() == "YES":
    idx_files = (
        glob.glob(f"{DRIVE_BASE}/data/processed/bm25_*.pkl")
        + glob.glob(f"{DRIVE_BASE}/data/processed/dense_*.faiss")
        + glob.glob(f"{DRIVE_BASE}/data/processed/dense_*.pkl")
    )
    csv_files = glob.glob(f"{DRIVE_BASE}/results/*.csv")
    for p in idx_files + csv_files:
        os.remove(p)
        print(f"Deleted: {p}")

    ckpt_dir = f"{DRIVE_BASE}/data/processed/checkpoints"
    for variant in ["v1", "v3"]:
        ckpt_path = f"{ckpt_dir}/{variant}"
        if os.path.exists(ckpt_path):
            shutil.rmtree(ckpt_path)
            print(f"Deleted checkpoint: {ckpt_path}")

    print(f"Removed {len(idx_files)} index files, {len(csv_files)} result CSVs, and checkpoints.")
    print("Now re-run Phase 2 (build indexes), Phase 3 (train V1/V3), and Phase 4 (evaluate).")
else:
    print("Aborted — nothing deleted.")